In [ ]:
# Dicas para executar notebooks no Google Colab:
# `pip install eegdash`
# Configura visualização gráfica inline no Jupyter notebook
%matplotlib inline

# Curvas de aprendizado a partir de sujeitos reais de treino

Como a adição de um participante de treinamento altera o desempenho na validação?
Usamos subconjuntos aninhados de sujeitos de treino e mantemos o sujeito 3 fixo para validação.
Dois tamanhos de treino ilustram o procedimento, não uma lei geral de escalonamento.

Dados: Nakanishi2015, NEMAR ``nm000118``, sujeitos 1–3, sessão 0,
execução 0: aproximadamente 21.1 MB no primeiro download. Defina ``EEGDASH_CACHE_DIR``
para reutilizar o cache. Esta versão processada já inclui filtragem,
redução da taxa de amostragem (*downsampling*) e tratamento de latência; não adicione outra correção de latência.
Consulte o [estudo de origem](https://doi.org/10.1371/journal.pone.0140703)
e a [versão NEMAR](https://nemar.org/dataset/nm000118).

Pré-requisitos: o ajuste e avaliação disjuntos por sujeitos do tutorial 51, além da
linha de base espectral do tutorial 12. Nenhum modelo salvo ou arquivo de saída anterior é
necessário. A saída é uma curva de validação de dois pontos, com a contagem de participantes
e a contagem de ensaios informadas para que o eixo horizontal seja inequívoco.


## 1. Selecionar uma coorte pequena e explícita
Filtrar sujeitos, sessão e execução delimita o download. Cortar (*crop*) após
abrir uma gravação reduziria a computação, mas não o tamanho do download.



In [ ]:
# Importa módulos de sistema operacional, funções parciais e caminhos de arquivo
import os
from functools import partial
from pathlib import Path

# Importa bibliotecas para plotagem, manipulação de arrays numéricos e DataFrames
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
# Importa gerador de janelas baseadas em eventos da Braindecode
from braindecode.preprocessing import create_windows_from_events
# Importa modelo de classificação e métricas do scikit-learn
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# Importa classes de dataset e extratores espectrais do EEGDash
from eegdash import EEGDashDataset
from eegdash.features import (
    FeatureExtractor,
    extract_features,
    spectral_bands_power,
    spectral_preprocessor,
)

# Define o caminho do diretório de cache
cache_dir = Path(os.environ.get("EEGDASH_CACHE_DIR", ".eegdash_cache"))
# Lista com os três sujeitos utilizados para curva de aprendizado
subjects = ["1", "2", "3"]
# Carrega o conjunto de dados da tarefa de SSVEP
dataset = EEGDashDataset(
    cache_dir=cache_dir,
    dataset="nm000118",
    subject=subjects,
    session="0",
    run="0",
    task="ssvep",
    n_jobs=1,
)
# Valida que há uma gravação por participante
assert len(dataset.datasets) == len(subjects), "Expected one recording per subject"
# Exibe resumo das gravações carregadas
print(dataset.description[["subject", "session", "run"]])

## 2. Inspecionar anotações reais e verificar o contrato do sinal
Acessar ``raw`` faz o download dessa gravação. Os nomes das anotações identificam a
frequência do estímulo atendido em Hz; eles fornecem cada rótulo de classificação.
Todos os participantes devem ter a mesma ordem de canais e frequência de amostragem.



In [ ]:
# Acessa os dados brutos do primeiro participante para extrair parâmetros
raw = dataset.datasets[0].raw
sfreq = raw.info["sfreq"]
channel_names = raw.ch_names
# Obtém as classes de frequência ordenadas numericamente
class_names = sorted(set(raw.annotations.description), key=float)
# Cria mapeamento da frequência textual para índice de 0 a 11
mapping = {name: index for index, name in enumerate(class_names)}
# Valida que existem as doze frequências de estímulo
assert len(mapping) == 12, "Expected the twelve SSVEP stimulus frequencies"
# Garante conformidade de canais, taxa de amostragem e anotações em todas as gravações
for recording in dataset.datasets:
    recording_raw = recording.raw
    assert recording_raw.ch_names == channel_names
    assert recording_raw.info["sfreq"] == sfreq
    assert set(recording_raw.annotations.description) == set(mapping)
# Imprime informações dos canais e classes encontradas
print(f"Channels: {channel_names}; sampling frequency: {sfreq} Hz")
print("Stimulus frequencies (Hz):", class_names)

## 3. Fazer uma janela de quatro segundos por ensaio anotado
Cada intervalo anotado dura 4.15 segundos. Mantenha seus primeiros quatro segundos e
descarte o restante. Tamanho e passo (*stride*) explícitos evitam janelas sobrepostas
ou a extensão da época além da duração do evento gravado.
A 256 Hz, quatro segundos contêm 1.024 amostras. O array resultante possui
eixos (540 ensaios, 8 canais de EEG, 1.024 amostras), com dados expressos em volts.
Os metadados possuem uma linha por linha do array. ``target`` é um índice de classe, não uma
frequência em Hz; ``mapping`` é a conversão explícita entre eles.
A fonte contém 15 ensaios de cada uma das 12 frequências por pessoa.
Uma classe ausente é uma falha no contrato de dados, não um motivo para re-rotular ensaios.



In [ ]:
# Define o tamanho da janela em amostras correspondente a 4 segundos (4 * 256 = 1024 amostras)
window_size = int(4 * sfreq)
# Cria as janelas baseadas nas pistas visuais descartando frações excedentes ao final
windows = create_windows_from_events(
    dataset,
    mapping=mapping,
    trial_start_offset_samples=0,
    trial_stop_offset_samples=0,
    window_size_samples=window_size,
    window_stride_samples=window_size,
    on_last_window="drop",
    preload=True,
)
# Extrai os metadados associados a cada janela
metadata = windows.get_metadata()
# Valida que há exatamente uma janela extraída por ensaio gravado
assert (metadata.i_window_in_trial == 0).all(), "Expected one window per trial"
# Confirma ausência de duplicatas de ensaios
assert not metadata.duplicated(["subject", "session", "run", "i_start_in_trial"]).any()
# Extrai o array numérico das classes alvo y
y = metadata["target"].to_numpy(dtype=int)
# Extrai o array com o identificador do sujeito de cada janela
groups = metadata["subject"].astype(str).to_numpy()
# Empilha os dados em uma matriz 3D (n_ensaios, n_canais, n_amostras)
X = np.stack([window[0] for window in windows])
# Valida dimensões esperadas, presença dos três sujeitos e finitude numérica
assert X.shape == (len(metadata), len(channel_names), window_size)
assert set(groups) == set(subjects)
assert np.isfinite(X).all()
# Exibe tabela cruzada de contagem de ensaios por sujeito e por classe
print(pd.crosstab(groups, y, rownames=["subject"], colnames=["class"]))

## 4. Extrair características espectrais de cada janela
As respostas de SSVEP contêm energia na frequência do estímulo. Use o log da potência espectral
ao redor de cada frequência de estímulo, mantendo todos os oito canais posteriores.
Esta transformação por janela não aprende nada a partir de outros ensaios ou sujeitos.
O escalonador abaixo, por outro lado, deve ser ajustado apenas nos sujeitos de treino.
O pré-processador espectral compartilhado do EEGDash calcula uma PSD de Welch com um segmento Hann
de quatro segundos e bins de 0.25 Hz. Cada banda estreita é centrada
em uma frequência de estímulo documentada; esses centros definem a tarefa, não
preditores específicos de ensaios. Reter oito canais fornece 12 × 8 = 96
características. A versão já processada de SSVEP não precisa de outro passe de limpeza
do EEGPrep ou correção de latência visual.

``spectral_bands_power`` soma os bins selecionados da PSD. Multiplicar pelo espaçamento
de 0.25 Hz converte V²/Hz para a potência aproximada da banda em V². O log comprime
essa escala; o StandardScaler ainda é ajustado apenas nos participantes de treino.



In [ ]:
# Define as bandas estreitas ao redor de cada frequência de estímulo (largura total de 0.25 Hz)
bands = {
    f"hz_{name}": (float(name) - 0.125, float(name) + 0.125) for name in class_names
}
# Configura o extrator com Welch PSD de 8 a 16 Hz e resolução de 0.25 Hz
spectral = FeatureExtractor(
    {"power": partial(spectral_bands_power, bands=bands)},
    preprocessor=partial(
        spectral_preprocessor,
        fs=sfreq,
        nperseg=window_size,
        noverlap=0,
        f_min=8,
        f_max=16,
    ),
)
# Extrai características espectrais para todas as janelas
feature_table = extract_features(
    windows, {"spectral": spectral}, batch_size=64, n_jobs=1
).to_dataframe()
# Valida o número de características calculadas (12 classes * 8 canais = 96 colunas)
assert feature_table.shape == (len(y), len(class_names) * len(channel_names))
# Converte densidade para potência em V² multiplicando pelo bin width (0.25 Hz) e aplica transformação log
features = np.log(np.maximum(feature_table.to_numpy() * sfreq / window_size, 1e-30))
# Garante que todos os valores obtidos são finitos
assert np.isfinite(features).all()

## 5. Expandir uma coorte de treino aninhada; manter a validação fixa
A permutação com semente define a ordem dos sujeitos, sem nunca fabricar observações.
Inspeções repetidas tornam o sujeito 3 um conjunto de validação, não um conjunto de teste final.
Um estudo maior necessitaria de sujeitos de teste adicionais intocados e ordens repetidas.
Ambos os subconjuntos de treinamento contêm todas as classes de frequência. O subconjunto de um participante
contribui com 180 ensaios; o subconjunto de dois participantes contém esses mesmos ensaios
mais os 180 ensaios da outra pessoa. Esse aninhamento impede que uma mudança na composição
do treino seja confundida com uma mudança de tamanho.
O sujeito 3 fornece os mesmos 180 ensaios de validação em ambos os pontos.

A extração de características é fixa, mas o escalonador e o classificador são reajustados
para cada tamanho. Reutilizar o ajuste maior permitiria que informações do participante adicionado
entrassem no resultado de tamanho menor.



In [ ]:
# Inicializa gerador de números pseudoaleatórios com semente fixa para reprodutibilidade
rng = np.random.default_rng(42)
# Permuta aleatoriamente a ordem dos sujeitos de treino 1 e 2
training_order = rng.permutation(["1", "2"])
# Define os índices fixos do sujeito 3 que servirá exclusivamente como conjunto de validação
validation = np.flatnonzero(groups == "3")
# Inicializa lista de registros e conjunto de controle de aninhamento
rows = []
previous = set()
# Itera incrementando o tamanho da coorte de treino: 1 sujeito e depois 2 sujeitos
for n_subjects in (1, 2):
    # Obtém índices correspondentes aos primeiros n_subjects da ordem sorteada
    train = np.flatnonzero(np.isin(groups, training_order[:n_subjects]))
    # Assegura que o subconjunto anterior está estritamente contido no atual (aninhamento cumulativo)
    assert previous.issubset(set(train))
    # Assegura que o conjunto de treino e validação são estritamente disjuntos
    assert set(groups[train]).isdisjoint(groups[validation])
    # Assegura presença das doze classes tanto no treino quanto na validação
    assert set(y[train]) == set(y[validation]) == set(mapping.values())
    # Atualiza o conjunto de índices anteriores
    previous = set(train)
    # Constrói o pipeline contendo padronizador e regressão logística
    model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
    # Ajusta o modelo usando estritamente os ensaios do subconjunto de treino atual
    model.fit(features[train], y[train])
    # Avalia o modelo nos 180 ensaios de validação do sujeito 3
    prediction = model.predict(features[validation])
    # Registra o número de sujeitos, número total de ensaios e a acurácia balanceada obtida
    rows.append(
        dict(
            n_subjects=n_subjects,
            n_trials=len(train),
            balanced_accuracy=balanced_accuracy_score(y[validation], prediction),
        )
    )
# Constrói e exibe a tabela da curva de validação
results = pd.DataFrame(rows)
print("Training order:", training_order)
print(results.to_string(index=False))

## 6. Exibir pontuações medidas de validação sem extrapolação



In [ ]:
# Plota a curva de validação: número de participantes de treino vs. acurácia balanceada
plt.plot(results.n_subjects, results.balanced_accuracy, "o-")
# Traça linha tracejada horizontal com a taxa de acerto ao nível do acaso (1/12 ≈ 8.33%)
plt.axhline(1 / len(mapping), color="black", linestyle="--", label="Chance (1/12)")
# Configura marcas e rótulos dos eixos
plt.xticks([1, 2])
plt.xlabel("Training participants")
plt.ylabel("Subject 3 validation balanced accuracy")
plt.ylim(0, 1)
plt.legend()
# Exibe o gráfico
plt.show()

## 7. Ler uma pequena curva de aprendizado sem superinterpretá-la
Um segmento ascendente indica melhora neste único participante de validação
para esta ordem específica de sujeitos de treino. Um segmento plano ou descendente
é igualmente válido: adicionar um participante altera tanto o tamanho quanto a composição da população.
Não há barras de erro porque apenas uma ordem foi avaliada.

Com uma coorte maior, repita várias ordens aninhadas de sujeitos de treino mantendo
as identidades de validação fixas e relate a dispersão em cada tamanho.
Reserve pessoas adicionais intocadas para o modelo final escolhido. Não
extrapole uma lei geral de eficiência amostral a partir destes dois pontos.

